# 4. Аудит архива и выбор протокола оценки

Вид спорта имеет шесть классов; пол и возраст — по два. Возраст `under_18` означает соревнование до 18 лет, `adult_18_plus` — взрослую категорию от 18. Метки относятся к соревнованию.

Группа `source_group_id` объединяет окна одного видео и записи с одинаковым нормализованным названием. Канал — публикующий аккаунт. Несколько окон одной группы дают один итоговый прогноз после усреднения логитов. Сначала использовалось исходное разбиение архива, затем проверка стала строже: каналы не пересекаются между train, validation и test.

Код: `make_vk_owner_splits.py`, аудит и применение исправлений внутри `train_vk_bundle.py`.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/train_vk_bundle.py").is_file()
             and (p / "configs/experiment_index.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите ноутбук внутри sports_video_repo")

def load_json(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8"))

def load_jsonl(path):
    return [json.loads(line) for line in (ROOT / path).read_text(encoding="utf-8").splitlines() if line]

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
print("Читаем сохранённые результаты; обучение и скачивание не запускаются.")

Читаем сохранённые результаты; обучение и скачивание не запускаются.


In [2]:
parts = ("train", "validation", "test")
original = {p: load_jsonl(f"sports_training_bundle/{p}.jsonl") for p in parts}
holdout = {p: load_jsonl(f"data/splits/vk_owner_holdout/{p}.jsonl") for p in parts}
def owner(row):
    return row["video_id"].split("_", 1)[0]
def inventory(splits):
    return pd.DataFrame([{ "часть": part, "окон": len(rows),
       "видео": len({r["video_id"] for r in rows}), "групп": len({r["source_group_id"] for r in rows}),
       "каналов": len({owner(r) for r in rows})} for part, rows in splits.items()])
display(Markdown("**Исходное разбиение**")); display(inventory(original))
display(Markdown("**Разбиение по каналам**")); display(inventory(holdout))
train_owners = {owner(r) for r in original["train"]}
familiar_test = {r["source_group_id"] for r in original["test"] if owner(r) in train_owners}
print("Тестовых групп от знакомых train-каналов в исходном split:", len(familiar_test))
for i, a in enumerate(parts):
    for b in parts[i+1:]:
        for key in ("video_id", "source_group_id"):
            assert not ({r[key] for r in holdout[a]} & {r[key] for r in holdout[b]})
        assert not ({owner(r) for r in holdout[a]} & {owner(r) for r in holdout[b]})
print("Пересечений видео, групп и публикующих каналов в holdout нет.")

**Исходное разбиение**

,часть,окон,видео,групп,каналов
0,train,720,269,263,113
1,validation,179,67,66,46
2,test,187,68,67,45


**Разбиение по каналам**

,часть,окон,видео,групп,каналов
0,train,766,284,277,89
1,validation,155,59,58,26
2,test,165,61,61,23


Тестовых групп от знакомых train-каналов в исходном split: 53
Пересечений видео, групп и публикующих каналов в holdout нет.


## Исправления и распределение независимых групп

In [3]:
corrections = load_json("data/metadata/vk_visual_corrections.json")["sport_by_video_id"]
print("Вручную исправленных видео:", len(corrections))
print("Затронутых окон:", sum(r["video_id"] in corrections for rows in original.values() for r in rows))
summary = load_json("data/splits/vk_owner_holdout/summary.json")
display(pd.DataFrame({p: summary["splits"][p]["sport_groups"] for p in parts}).fillna(0).astype(int))
cells = []
for part, rows in holdout.items():
    groups = {r["source_group_id"]: r for r in rows}
    for r in groups.values():
        cells.append({"часть": part, "спорт": corrections.get(r["video_id"], r["sport"]),
                      "пол": r["gender"], "возраст": r["age"]})
display(pd.DataFrame(cells).groupby(["спорт", "пол", "возраст", "часть"]).size().unstack("часть", fill_value=0))

Вручную исправленных видео: 10
Затронутых окон: 25


,train,validation,test
basketball,32,13,13
football,78,15,7
tennis,54,10,16
table_tennis,39,5,8
handball,26,6,8
volleyball,48,9,9


часть                             test  train  validation
спорт        пол   возраст                               
basketball   men   adult_18_plus     0      7           3
                   under_18          8     17           1
             women adult_18_plus     2      5           0
                   under_18          3      3           9
football     men   adult_18_plus     0      6           4
                   under_18          7     51           6
             women adult_18_plus     0      9           5
                   under_18          0     12           0
handball     men   adult_18_plus     2      6           0
                   under_18          1      5           2
             women adult_18_plus     5     12           1
                   under_18          0      3           3
table_tennis men   adult_18_plus     5      8           4
                   under_18          0      5           0
             women adult_18_plus     3     15           1
                   under_18          0     11           0
tennis       men   adult_18_plus     5     29           3
             women adult_18_plus    10     24           7
                   under_18          1      1           0
volleyball   men   adult_18_plus     1      3           3
                   under_18          4     19           0
             women adult_18_plus     2      8           5
                   under_18          2     18           1

## Что значит число кадров

Для окна обычно сохранено около 25 JPEG. DINOv2 1f использует один центральный кадр; 4f — четыре распределённых кадра с усреднением признаков. Для видео ответы от 2–3 окон усредняются. В финальном X3D-XS четыре кадра выбраны около середины с шагом примерно 0,4 с; R3D-18 получает 16 кадров по всему окну.

Основной размер входа: ResNet-18 и DINOv2 — 224×224, X3D — 160×160, R3D — 112×112. Архивные кадры обычно 480×270. Только визуальная часть подаётся в классификатор; текст используется для подготовки меток.

Лучшая эпоха выбирается по среднему Macro-F1 трёх голов на validation. Test используется для оценки. Эти разделения снижают риск подсказок от знакомого источника, но не доказывают, что разные загрузки всегда относятся к разным турнирам. Метки теста остаются слабыми.